# Chapter 32 | Worked solutions

*AI Agents with Language Models*

Follow the same numbering as the [exercise sheet](../../examples/ch32_agents/EXERCISES.md).
The campus desk answers questions about fictional workshops. The recorded
gpt-oss-20b runs let us follow the evidence from the first catalog lookup
to a checked quote, a request for clarification, or a stop.

## Setup

Run all cells in order from the repository root or this notebook's folder.
The Python cells use the standard library and the chapter's saved JSON
records. Reading the records and running the deterministic checks requires
no model download. Exercise 10 uses an in-memory reservation example.

In [1]:
import json
from math import isclose, sqrt
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "examples/ch32_agents/records").is_dir()),
    None,
)
if root is None:
    raise FileNotFoundError("Open the notebook inside transformers-book.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from examples.ch32_agents.campus import CASES, dispatch, quote_total
from examples.ch32_agents.fixed_workflow import decision, run_case

records = root / "examples/ch32_agents/records"

def load_record(name):
    return json.loads((records / name).read_text(encoding="utf-8"))

measured = {row["case"]: row for row in load_record("MEASURED_RUN.json")["runs"]}
print(f"Loaded {len(measured)} recorded campus-desk runs.")

Loaded 6 recorded campus-desk runs.


## 1. Trace a changed room

Start with six tool calls available. Each dispatched call uses one slot.
The trace grows in order, so the history after step 3 includes the first
three observations.

| Call | Active notice | Cancelled notice |
| --- | --- | --- |
| 1. Catalog | Camera Lab is E-CAM. The catalog lists R-OAK, 800 cents per person, a 200-cent group fee, and notice N-17. Five calls remain. | Same catalog result. Five remain. |
| 2. Notice | N-17 is active. It gives R-MAPLE and a new rate of 875 cents. Four remain. | N-17 reports cancellation. Four remain. |
| 3. Room | R-MAPLE is open and has room for 16 people. Three remain. | The model returns a cancellation answer. Four remain. |
| 4. Quote | The calculator returns 10,700 cents. Two remain. The final answer is a quote for R-MAPLE. | The run has ended. |

The first different observation is the result for N-17. The next model
turn proposes a room lookup in the active case and returns a final answer
in the cancelled case. A final answer consumes no tool-call slot.

In [2]:
active = measured["camera_updated"]
cancelled = measured["camera_cancelled"]

for run in (active, cancelled):
    remaining = 6
    print(run["case"])
    for step in run["trace"]:
        if step["event"] == "tool_result":
            remaining -= 1
            print(f"  {step['tool']:<12}  {remaining} remaining")
        elif step["event"] == "final":
            print(f"  final {run['final']['status']}")
    assert remaining == 6 - run["executed_calls"]

assert active["trace"][0]["observation"] == cancelled["trace"][0]["observation"]
assert active["trace"][1]["observation"]["status"] == "active"
assert cancelled["trace"][1]["observation"]["status"] == "cancelled"
assert active["trace"][2]["tool"] == "room_status"
assert cancelled["trace"][2]["event"] == "final"

camera_updated
  find_session  5 remaining
  read_notice   4 remaining
  room_status   3 remaining
  quote_total   2 remaining
  final fits
camera_cancelled
  find_session  5 remaining
  read_notice   4 remaining
  final cancelled


## 2. Check arguments and evidence

Use 12 guests and a 200-cent group fee. The dispatcher expects integers
with the exact field names in the tool schema.

| Proposal for the calculator | What happens |
| --- | --- |
| The party size is the string `"12"` | The value fails the integer check. |
| All the required arguments plus an extra field | The set of fields fails validation. |
| Integer arguments with a rate of 800 cents | The call passes validation and returns 9,800 cents. The rate must still be verified. |

The arithmetic in the third proposal is valid for its inputs.
The catalog's rate was 800 cents. The active notice N-17 supersedes it with
875 cents, so the checked quote is $12(875)+200=10{,}700$ cents.
The schema validates the shape and type of a request. The notice establishes
which rate applies.

In [3]:
older_quote = {"party_size": 12, "rate_cents": 800, "fee_cents": 200}
proposals = {
    "string party size": dict(older_quote, party_size="12"),
    "extra field": dict(older_quote, currency="USD"),
    "old rate": older_quote,
}
results = {
    label: dispatch("quote_total", arguments, cancelled_camera=False)
    for label, arguments in proposals.items()
}
for label, result in results.items():
    print(f"{label:<18} {result}")

notice = dispatch("read_notice", {"notice_id": "N-17"}, cancelled_camera=False)
current_quote = dict(older_quote, rate_cents=notice["rate_cents"])
verified = dispatch("quote_total", current_quote, cancelled_camera=False)
print(f"rate from {notice['id']}     {verified}")
assert results["string party size"] == {"error": "integer_out_of_range"}
assert results["extra field"] == {"error": "wrong_fields"}
assert results["old rate"]["total_cents"] == 9_800
assert verified["total_cents"] == 10_700

string party size  {'error': 'integer_out_of_range'}
extra field        {'error': 'wrong_fields'}
old rate           {'total_cents': 9800}
rate from N-17     {'total_cents': 10700}


## 3. Calculate an exact price

The seven per-person charges total $7(675)=4{,}725$ cents. Adding the
group fee once gives $4{,}725+250=4{,}975$ cents, or **49.75 dollars**.
Charging the fee for each person gives $7(675+250)=6{,}475$ cents and
adds six extra fees, or $6(250)=1{,}500$ cents. The tool's fee is for the whole
group. Keep every amount in integer cents until the final display.

In [4]:
guest_count, rate_cents, fee_cents = 7, 675, 250
total_cents = quote_total(guest_count, rate_cents, fee_cents)["total_cents"]
dollars, cents = divmod(total_cents, 100)
per_person_fee_total = guest_count * (rate_cents + fee_cents)
extra_cents = per_person_fee_total - total_cents

print(f"Group fee once   {total_cents} cents, ${dollars}.{cents:02d}")
print(f"Fee per person   {per_person_fee_total} cents")
print(f"Extra cost       {extra_cents} cents")
assert (total_cents, extra_cents) == (4_975, 1_500)

Group fee once   4975 cents, $49.75
Fee per person   6475 cents
Extra cost       1500 cents


## 4. Budget a conversation

Use $B=960$ tokens in the first call and add $d=180$ tokens between
successive calls. The five prompt lengths are 960, 1,140, 1,320, 1,500, and
1,680 tokens. Summing them gives 6,600. In the formula, the original context
is counted five times and the increments form the sum $0+1+2+3+4=10$.

$$
mB + d\frac{m(m-1)}{2}
=5(960)+180\frac{5(4)}{2}=6{,}600.
$$

The 1,600-token limit applies to each prompt. Call 4 fits at 1,500 tokens.
Call 5 needs compression or a shorter context by at least 80 tokens.

In [5]:
base_tokens, added_per_call, calls, input_cap = 960, 180, 5, 1_600
lengths = [base_tokens + added_per_call * index for index in range(calls)]
closed_form = calls * base_tokens + added_per_call * calls * (calls - 1) // 2
first_over = next(index for index, count in enumerate(lengths, 1) if count > input_cap)

print("Prompt lengths", lengths)
print("Total input tokens", sum(lengths))
print(f"Call {first_over} exceeds the cap by {lengths[first_over - 1] - input_cap}.")
assert sum(lengths) == closed_form == 6_600
assert first_over == 5

Prompt lengths [960, 1140, 1320, 1500, 1680]
Total input tokens 6600
Call 5 exceeds the cap by 80.


## 5. Design a workflow baseline

Start with a supplied title and party size. At each step, a definite
result selects one branch.

| Step | Outcome or next step |
| --- | --- |
| Find the title | Ask when multiple sessions match. Report missing evidence if none match. Continue with a unique session. |
| Read the listed revision | Stop on cancellation. Apply the new room and rate when the notice supplies them. Continue with the current room. |
| Check that room | Report unavailable if closed, or too large if the party exceeds capacity. Continue when the group fits. |
| Calculate the price | Multiply the verified rate by party size and add the catalog fee once. Return the amount and the applicable record IDs. |

For a comparison on the same free-text requests, put an extraction stage
before the deterministic workflow. It supplies the title and party size,
or asks for a missing or ambiguous field. Grade extraction separately and
include its errors, calls, and time in the end-to-end comparison. The
published baseline starts with the two fields already supplied.

In [6]:
cases = {case.name: case for case in CASES}
for name in ("sketch_ambiguous", "camera_cancelled", "paper_too_large", "ink_fits"):
    record = run_case(cases[name])
    assert record["success"]
    print(f"{name:<18} {record['final']['status']:<10} {record['executed_calls']} calls")

def closed_room(tool, **arguments):
    data = dispatch(tool, arguments, cancelled_camera=False)
    return dict(data, open=False) if tool == "room_status" else data

status, room, price, _, _ = decision(cases["ink_fits"], closed_room)
assert (status, room, price) == ("unavailable", "R-MAPLE", None)
print(f"{'closed-room fixture':<18} {status}")

sketch_ambiguous   ask        1 calls
camera_cancelled   cancelled  2 calls
paper_too_large    too_large  2 calls
ink_fits           fits       3 calls
closed-room fixture unavailable


## 6. Make one instruction more precise

The first Sketch Club run returns `status="ask"` and names the two
sessions in its message. Its structured `evidence` list is empty,
so the choice is legible to a person but the program has no IDs in that
field. The revised instruction asks the model to copy both candidate IDs
from the catalog result into `evidence`. The returned IDs are E-SK1
and E-SK2. A program can now join them to their catalog rows and wait for
the person's selection. Both runs leave the room and total unspecified
until the session is chosen.

A second case could ask about *Pottery Studio* and return Morning Pottery
Studio (E-POT1) and Evening Pottery Studio (E-POT2). The expected response
asks which session and carries both IDs in `evidence`. This tests whether
the instruction works when the candidate names and IDs change.

In [7]:
original_sketch = measured["sketch_ambiguous"]
revised_sketch = load_record("CLARIFICATION_RUN.json")["runs"][0]
candidates = original_sketch["trace"][0]["observation"]["matches"]
candidate_ids = [candidate["id"] for candidate in candidates]

for label, run in (("original", original_sketch), ("revised", revised_sketch)):
    final = run["final"]
    assert final["status"] == "ask"
    assert final["room_id"] is None and final["total_cents"] is None
    print(f"{label:<10} {final['status']}   evidence {final['evidence']}")
assert original_sketch["final"]["evidence"] == []
assert revised_sketch["final"]["evidence"] == candidate_ids == ["E-SK1", "E-SK2"]

original   ask   evidence []
revised    ask   evidence ['E-SK1', 'E-SK2']


## 7. Report a truncated run

The two executed calls are the catalog search and the notice lookup.
The agent next proposes `room_status` for R-MAPLE. The runner logs the
proposal at the limit and stops before dispatch. The missing observation
is whether the current room is open and whether its capacity is at
least 12. Once that is known, the calculator can use the current rate
and one group fee.

A response at the stop could say, “The latest notice moves Camera Lab
to the Maple Room. I still need to check whether it is open and has
room for your group before I can confirm a quote.”

In [8]:
limited = load_record("BUDGET_RUN.json")["runs"][0]
executed = [step["tool"] for step in limited["trace"] if step["event"] == "tool_result"]
blocked = next(step for step in limited["trace"] if step["event"] == "tool_limit")
print(f"Executed {limited['executed_calls']} calls   {', '.join(executed)}")
print(f"Proposed at the limit   {blocked['tool']} for {blocked['arguments']['room_id']}")
assert executed == ["find_session", "read_notice"]
assert limited["executed_calls"] == 2
assert blocked["tool"] == "room_status" and blocked["arguments"]["room_id"] == "R-MAPLE"
assert limited["final"] is None

Executed 2 calls   find_session, read_notice
Proposed at the limit   room_status for R-MAPLE


## 8. Check reliability across runs

With independent attempts and success probability $0.7$, the probability
of three failures is $(1-0.7)^3=0.027$. Its complement gives the probability
of at least one success, $1-0.027=0.973$. All three succeed with probability
$0.7^3=0.343$. A shared catalog outage can block every attempt. In that
case the attempts share a failure cause, so multiplying the marginal
probabilities gives the wrong joint probabilities.

In [9]:
p_success, attempts = 0.7, 3
at_least_one = 1 - (1 - p_success) ** attempts
all_three = p_success ** attempts
print(f"At least one success   {at_least_one:.3f}")
print(f"All three succeed      {all_three:.3f}")
assert isclose(at_least_one, 0.973) and isclose(all_three, 0.343)

At least one success   0.973
All three succeed      0.343


## 9. Plan a handoff

Suppose the user needs a step-free entrance for Camera Lab. Send the
confirmed event and room, the session time, the two accessibility
records with their sources and dates, and the conflicting entrance
details. The question for the specialist is which entrance is accessible
for this session.

| Pattern | Message sequence | Owner of the next decision |
| --- | --- | --- |
| Tool-style delegation | Lead sends the event, both records, and the question to the specialist. Specialist returns an entrance and the evidence that confirms it, or reports the unresolved conflict. | The lead decides how to answer or what to check next. |
| Handoff | Lead transfers the user goal, verified facts, both records, and the unresolved question. Specialist picks up the session, checks the current record, and answers or asks for missing evidence. | The specialist owns the next tool request or answer after receiving control. |

In either pattern, the specialist can see which facts were already
verified and which question remains open.

## 10. Verify effects after a timeout

Before repeating a reservation request, look up the operation ID and
compare the stored event and party size with the intended reservation.
A committed operation should expose its reservation ID or receipt. A
pending or uncertain result calls for another status check. A definitive
absence allows a retry under a supported idempotency contract, using
the same operation ID. If the ID points to different inputs, inspect
the mismatch before proceeding.

In an acceptance test, apply the reservation then drop the response.
Look up the operation, read its receipt, and exercise a retry with the
same ID and arguments. The final simulated store must contain one
reservation for the operation. Its record ID, event, and party size
must agree with the first receipt.

In [10]:
reservations = {}

def submit_reservation(operation_id, event_id, party_size, *, lose_reply=False):
    request = {"event_id": event_id, "party_size": party_size}
    if operation_id not in reservations:
        reservations[operation_id] = {
            **request, "reservation_id": f"RES-{len(reservations) + 1:03d}"
        }
    receipt = reservations[operation_id]
    if any(receipt[field] != value for field, value in request.items()):
        raise ValueError("Operation ID was already used with different inputs.")
    if lose_reply:
        raise TimeoutError("Response lost after the simulated commit.")
    return receipt.copy()

operation = ("reserve-1", "E-CAM", 12)
try:
    submit_reservation(*operation, lose_reply=True)
except TimeoutError:
    receipt = reservations[operation[0]].copy()
else:
    raise AssertionError("The fixture should drop the first response.")

retried = submit_reservation(*operation)
assert retried == receipt
assert len(reservations) == 1
assert (receipt["event_id"], receipt["party_size"]) == operation[1:]
print(f"One reservation   {receipt['reservation_id']} for {receipt['party_size']} guests")

One reservation   RES-001 for 12 guests


## 11. Recover a waiting task

The checkpoint at simulated minute 0 has the user's goal, event E-CAM,
notice pointer N-17, the pending result, and the history of the two calls.
It also has the wake time (minute 180), deadline (minute 240), and the
remaining limits derived from the attempt and turn counts. Those facts
let a fresh process recover the task and know when it can resume.

| Continuation | What happens after waking at minute 180 |
| --- | --- |
| Unguarded | The model looks up R-OAK using the old catalog room. It then sees the new notice and computes 10,700 cents for R-MAPLE. The trace lacks a check of R-MAPLE, so the runner marks the answer unverified. |
| Guarded | The runner rejects the R-OAK proposal until N-17 is read again. The new notice supplies R-MAPLE and 875 cents per guest. The agent then verifies R-MAPLE and obtains the price. The runner marks the answer complete. |

The first room lookup that loses its relevance is the proposed lookup
of R-OAK. The saved pending notice establishes when to wake. After the
wake time, read the now-current notice, check its room for availability
and capacity, then calculate the total with its rate and the catalog fee.
The two final totals happen to match. The trace distinguishes a verified
answer from one that used the wrong room check.

In [11]:
horizon = load_record("HORIZON_RUN.json")
waiting = horizon["waiting"]
print(f"Wake at {waiting['wake_at']}, deadline {waiting['deadline']}, "
      f"{waiting['executed_calls']} calls saved.")
for label in ("baseline", "guarded"):
    run = horizon[label]
    room_events = [
        (step["event"], step["arguments"]["room_id"])
        for step in run["trace"] if step.get("tool") == "room_status"
    ]
    print(label, run["status"], room_events, run["final"]["total_cents"])

assert waiting["status"] == "waiting" and waiting["wake_at"] == 180
assert horizon["baseline"]["status"] == "unverified"
assert horizon["guarded"]["status"] == "complete"
assert horizon["guarded"]["tool_attempts"] == horizon["guarded"]["executed_calls"] + 1
assert horizon["baseline"]["final"]["total_cents"] == horizon["guarded"]["final"]["total_cents"] == 10_700

Wake at 180, deadline 240, 2 calls saved.
baseline unverified [('tool_result', 'R-OAK')] 10700
guarded complete [('proposal_rejected', 'R-OAK'), ('tool_result', 'R-MAPLE')] 10700


## 12. Separate elapsed time from the measured horizon

Set $q=0.9$ in the human-reference success curve and solve for the
nonnegative value of $h$ in minutes.

$$
q=\frac{1}{1+(h/60)^2}
\quad\Longrightarrow\quad
h=60\sqrt{\frac{1}{q}-1}
=60\sqrt{\frac{1}{9}}=20.
$$

The benchmark horizon at 90 percent is 20 minutes of human reference
work. The separate run spends 20 minutes at the desk, 90 waiting,
and 15 verifying. Its wall-clock time is $20+90+15=125$ minutes.
The benchmark reports the human reference duration at the stated
success threshold. The elapsed time describes the actual run.

In [12]:
threshold, scale_minutes = 0.9, 60
horizon_minutes = scale_minutes * sqrt(1 / threshold - 1)
elapsed_minutes = sum((20, 90, 15))
assert isclose(1 / (1 + (horizon_minutes / scale_minutes) ** 2), threshold)
assert isclose(horizon_minutes, 20) and elapsed_minutes == 125
print(f"90% human reference horizon   {horizon_minutes:.0f} minutes")
print(f"Elapsed time for this run      {elapsed_minutes} minutes")

90% human reference horizon   20 minutes
Elapsed time for this run      125 minutes


## 13. Design a measurement

Create two types of held-out pair. For a new workshop called Lantern
Studio, keep the wording and party of 11 fixed. Give one run a notice
moving it to an open room for 16 and give the other a cancellation.
In a second pair, keep the active records fixed and vary the wording.
For example, compare “Can eleven of us go to Lantern Studio?” and
“Is there space for our group of 11 at Lantern Studio?” Ask for the
group price in both. Keep the underlying event and each family of
paraphrases together when splitting development and test cases.

| Measure | Rule |
| --- | --- |
| Outcome | Require the correct status, room when applicable, and exact group price when the party fits. Include stops and failures in the denominator. |
| Trace | For a quote, require the current notice when listed, verification of the current room, and a calculator call with the verified rate and fee in that order. Check that cancellation ends the run. |
| Cost | Report median tool attempts and median elapsed time, including waits. Keep rejected attempts visible alongside dispatched calls. |

The paired cases help identify which change led to a different answer.
A broader sample of ordinary requests estimates how often these
conditions and failures occur in use.